In [ ]:
from pathlib import Path

from utils import

ModuleNotFoundError: No module named 'shap'

# Configuration

In [ ]:
DATA_DIR = Path("output")
CSV_PATTERN = "dataset_winsize*.csv"

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

TARGETS = {
    "label_time_to_event_seconds": "regression",
    "label_from_zone": "classification",
    "label_to_zone": "classification",
}

DROP_COLUMNS = [
    "fold_id",
    "label_is_auto",
    "label_time_to_event_seconds",
    "label_from_zone",
    "label_to_zone",
]

In [ ]:
# -----------------------------
# Main loop over datasets
# -----------------------------

all_results = []

csv_files = sorted(DATA_DIR.glob(CSV_PATTERN))

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No files found matching {CSV_PATTERN} in {DATA_DIR.resolve()}"
    )

for csv_path in csv_files:
    dataset_name = csv_path.stem
    print(f"\nReading {csv_path}")

    df = pd.read_csv(csv_path)

    for target_col, task_type in TARGETS.items():
        if target_col not in df.columns:
            print(f"Skipping missing target: {target_col}")
            continue

        metrics_df = train_models_for_target(
            df=df,
            dataset_name=dataset_name,
            target_col=target_col,
            task_type=task_type,
        )

        all_results.append(metrics_df)

final_results = pd.concat(all_results, ignore_index=True)
final_results.to_csv(OUTPUT_DIR / "all_metrics_summary.csv", index=False)

print("\nFinal summary:")
print(final_results)